# Hafta 7 — Sınıflandırma II ve Özellik Mühendisliği

Ham titreşim sinyallerinden özellik çıkarıp karar ağacı, rastgele orman ve SVM ile arıza tipini sınıflandırıyoruz. Veri: `rulman_titresim.csv` (12 kHz, 2048 nokta × 480 parça, 4 sınıf).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, ConfusionMatrixDisplay

sig = pd.read_csv("rulman_titresim.csv")          # ham sinyaller: sinif + x0..x2047
fs, N = 12000, 2048
print(sig.shape); print(sig.sinif.value_counts())

## 1. Sinyallere bakalım

In [ ]:
t = np.arange(N) / fs
fig, ax = plt.subplots(2, 4, figsize=(15, 5))
for j, s in enumerate(["saglam", "ic_bilezik", "dis_bilezik", "bilye"]):
    x = sig[sig.sinif == s].iloc[0, 1:].values.astype(float)
    ax[0, j].plot(t[:600]*1000, x[:600], lw=.8); ax[0, j].set_title(s); ax[0, j].set_xlabel("ms")
    F = np.abs(np.fft.rfft(x)); fr = np.fft.rfftfreq(N, 1/fs); ax[1, j].plot(fr, F/F.max(), lw=.8); ax[1, j].set_xlim(0, 6000); ax[1, j].set_xlabel("Hz")
plt.tight_layout(); plt.show()

**Soru:** Arızalı sinyallerde zaman alanında ne görüyorsunuz? Spektrumda hangi bant güçleniyor?

## 2. Özellik çıkarımı: 2048 sayı → 10 sayı (Örnek 7.3)

In [ ]:
kurt = lambda x: np.mean((x - x.mean())**4) / np.var(x)**2
print("kurtosis:", kurt(np.array([1, -1, 1, -1, 1, -1, 1, -1.])), kurt(np.array([0, 0, 0, 4, 0, 0, 0, -4.])))

In [ ]:
def ozellik_cikar(x, fs=12000):
    """Tek bir sinyal parçasından (1-B dizi) özellik vektörü."""
    x = np.asarray(x, float); xm = x - x.mean()
    rms = np.sqrt(np.mean(x**2)); tepe = np.abs(x).max()
    kurt = np.mean(xm**4) / np.var(x)**2; carp = np.mean(xm**3) / np.std(x)**3
    F = np.abs(np.fft.rfft(x)); fr = np.fft.rfftfreq(len(x), 1/fs); top = F.sum()
    bant = lambda a, b: F[(fr >= a) & (fr < b)].sum() / top
    return {"rms": rms, "tepe": tepe, "crest": tepe/rms, "kurtosis": kurt, "carpiklik": carp,
            "bant_0_500": bant(0, 500), "bant_500_2k": bant(500, 2000), "bant_2k_3k": bant(2000, 3000), "bant_3k_4k": bant(3000, 4000), "bant_4k_6k": bant(4000, 6000)}

X_ham = sig.drop(columns="sinif").values
feat = pd.DataFrame([ozellik_cikar(x) for x in X_ham]); feat["sinif"] = sig.sinif.values
feat.head()

In [ ]:
# Sınıflara göre özellik dağılımları
fig, ax = plt.subplots(1, 4, figsize=(15, 3))
for a, k in zip(ax, ["rms", "kurtosis", "crest", "bant_3k_4k"]):
    for s in feat.sinif.unique(): a.hist(feat.loc[feat.sinif == s, k], bins=20, alpha=.5, label=s)
    a.set_title(k)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 3. Ayrım ve taban: lojistik regresyon

In [ ]:
X = feat.drop(columns="sinif"); y = feat.sinif
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
def degerlendir(ad, model):
    model.fit(Xtr, ytr); tahmin = model.predict(Xte)
    print(f"{ad:30s} doğruluk={accuracy_score(yte, tahmin):.3f}  makro-F1={f1_score(yte, tahmin, average='macro'):.3f}")
    return model
lojistik = degerlendir("Lojistik regresyon", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)))

## 4. Karar ağacı: Gini elle (Örnek 7.1) ve derinlik etkisi

In [ ]:
def gini(sayilar):
    p = np.array(sayilar) / sum(sayilar); return 1 - (p**2).sum()
ebeveyn = gini([6, 4])
for ad, sol, sag in (("kurtosis>3.5", [2, 4], [4, 0]), ("sicaklik>65", [3, 3], [3, 1])):
    ag = sum(sol)/10*gini(sol) + sum(sag)/10*gini(sag); print(f"{ad:14s} ağırlıklı Gini={ag:.3f}  kazanç={ebeveyn-ag:.3f}")

In [ ]:
for d in (1, 2, 3, 5, 8, None):
    a = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    print(f"derinlik={str(d):4s} eğitim={accuracy_score(ytr, a.predict(Xtr)):.3f}  test={accuracy_score(yte, a.predict(Xte)):.3f}  CV={cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), Xtr, ytr, cv=cv).mean():.3f}")

In [ ]:
agac = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
plt.figure(figsize=(16, 6)); plot_tree(agac, feature_names=X.columns, class_names=agac.classes_, filled=True, rounded=True, fontsize=8); plt.show()

**Soru:** Kök düğümdeki soru hangi özellik? Derinlik arttıkça eğitim ve test doğruluğu nasıl ayrışıyor?

## 5. Rastgele orman ve özellik önemi

In [ ]:
orman = degerlendir("Rastgele orman (300 ağaç)", RandomForestClassifier(300, random_state=0, oob_score=True))
print("OOB doğruluk:", round(orman.oob_score_, 3))
onem = pd.Series(orman.feature_importances_, X.columns).sort_values()
onem.plot.barh(figsize=(6, 3.5)); plt.xlabel("özellik önemi"); plt.show()

In [ ]:
# Ağaç sayısının etkisi
for B in (1, 5, 20, 100, 300):
    s = cross_val_score(RandomForestClassifier(B, random_state=0), Xtr, ytr, cv=cv); print(f"B={B:3d}  CV doğruluk = {s.mean():.3f} ± {s.std():.3f}")

## 6. SVM ve GridSearchCV

In [ ]:
gs = GridSearchCV(make_pipeline(StandardScaler(), SVC()),
                  {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}, cv=cv, n_jobs=-1)
gs.fit(Xtr, ytr)
print("en iyi:", gs.best_params_, " CV doğruluk:", round(gs.best_score_, 3))
res = gs.cv_results_["mean_test_score"].reshape(4, 4)
plt.imshow(res, cmap="Blues"); plt.colorbar(); plt.xticks(range(4), [0.001, 0.01, 0.1, 1]); plt.yticks(range(4), [0.1, 1, 10, 100]); plt.xlabel("gamma"); plt.ylabel("C")
for i in range(4):
    for j in range(4): plt.text(j, i, f"{res[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.show()
svm = degerlendir("SVM RBF (GridSearch)", gs.best_estimator_)

## 7. Çok sınıflı karışıklık matrisi (Örnek 7.4)

In [ ]:
tahmin = orman.predict(Xte)
ConfusionMatrixDisplay.from_predictions(yte, tahmin, cmap="Blues"); plt.show()
print(classification_report(yte, tahmin, digits=3))

**Soru:** Hangi iki sınıf en çok karışıyor? Neden (fiziksel açıklama)? Hangi özellik bunu ayırt edebilirdi?

## 8. One-hot: motor verisine motor_tipi eklemek (Hafta 6 verisi)

In [ ]:
m = pd.read_csv("motor_ariza.csv")
sayisal = ["calisma_saati", "titresim_rms_mms", "titresim_kurtosis", "sicaklik_C", "akim_dengesizlik_pct"]; kategorik = ["motor_tipi"]
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(m[sayisal + kategorik], m.ariza, test_size=0.3, random_state=0, stratify=m.ariza)
on_isleme = ColumnTransformer([("num", StandardScaler(), sayisal), ("kat", OneHotEncoder(handle_unknown="ignore"), kategorik)])
from sklearn.metrics import roc_auc_score
for ad, kolon in (("sayısal only", sayisal), ("+ one-hot motor_tipi", sayisal + kategorik)):
    ct = ColumnTransformer([("num", StandardScaler(), sayisal)] + ([("kat", OneHotEncoder(), kategorik)] if kategorik[0] in kolon else []))
    p = make_pipeline(ct, LogisticRegression(max_iter=1000)).fit(Xm_tr[kolon], ym_tr).predict_proba(Xm_te[kolon])[:, 1]
    print(f"{ad:24s} AUC = {roc_auc_score(ym_te, p):.3f}")

## 9. Alıştırmalar

**Alıştırma 1.** {5S, 5A} ve {10S, 0A} düğümleri için Gini ve entropiyi hesaplayın (kod). {8S, 2A} düğümünü {8S,0A}+{0S,2A} olarak bölen bir sorunun kazancı nedir?

In [ ]:
# Alıştırma 1

**Alıştırma 2.** `min_samples_leaf` parametresini 1, 5, 10, 20 için tarayın (derinlik sınırsız). Eğitim/test doğruluğu nasıl değişiyor? Bu parametre `max_depth` ile aynı işi mi yapıyor?

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Zarf analizi özellikleri ekleyin: sinyali 2.5-4.2 kHz bant geçiren filtreden geçirin (`scipy.signal.butter` + `sosfiltfilt`), Hilbert zarfını alın (`np.abs(hilbert(x))`), zarfın FFT'sinde teorik arıza frekansları etrafındaki (BPFO ≈ 107, BSF ≈ 141, BPFI ≈ 162 Hz, ±8 Hz) enerji oranlarını üç yeni özellik yapın. Bilezik/bilye karışıklığı azalıyor mu?

In [ ]:
# Alıştırma 3

**Alıştırma 4.** Test sinyallerine σ = 0.3 beyaz gürültü ekleyip (özellikleri yeniden çıkarın) lojistik, orman ve SVM'nin makro-F1'ini karşılaştırın. Hangi özellikler gürültüden en çok etkileniyor (gürültülü/temiz ortalama oranı)?

In [ ]:
# Alıştırma 4

**Alıştırma 5.** `RandomizedSearchCV` ile orman için `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features` uzayında 20 rastgele deneme yapın; GridSearch'e göre kaç eğitim tasarruf ettiniz?

In [ ]:
# Alıştırma 5